# agent-reliability-lab: real-model smoke run (Qwen3-0.6B)

Runs the 20-scenario smoke suite under the three harness configs with a **real** local model instead of the scripted
reference policy. Free Colab GPU (T4) or CPU both work. On CPU expect it to be slow; the global request cap bounds the cost.

* No API keys are needed: the model is public and inference is local.
* The model runs with `enable_thinking=False`, and only tool calls and final answers are logged.
* Record the exact model revision you use in the results.

In [ ]:
!nvidia-smi || echo "no GPU: running on CPU"
!pip -q install "agent-reliability-lab[local-llm] @ git+https://github.com/Pranjulcr7/agent-reliability-lab@main"

In [ ]:
from huggingface_hub import HfApi
MODEL = "Qwen/Qwen3-0.6B"
REVISION = HfApi().model_info(MODEL).sha   # pin: this exact commit is recorded in the run metadata
print(MODEL, REVISION)

In [ ]:
!git clone -q https://github.com/Pranjulcr7/agent-reliability-lab && cd agent-reliability-lab && \
  arlab generate --out data/suite --instances 3 && \
  arlab run --split smoke --seeds 0 --model "transformers:{MODEL}@{REVISION}" \
    --max-total-requests 600 --out results/smoke-qwen3-0.6b

In [ ]:
import json
s = json.load(open("agent-reliability-lab/results/smoke-qwen3-0.6b/summary.json"))
for cfg, v in s["by_config"].items():
    print(cfg, v["success_rate"], v["success_ci95"], "dup", v["duplicate_side_effects"],
          "unauth", v["unauthorized_attempts"], v["unauthorized_executed"], "stops", v["stop_reasons"])

Download `results/smoke-qwen3-0.6b/` (summary, episodes, traces) and commit it with the revision above.
n=20 per config is a smoke test: report the Wilson intervals, not point estimates alone.